# Lot D · Gold sur Kaggle (suite) : textes libres, robustesse, RQ2

Trois calculs GPU du palier Gold, tous sur le **dev** (test gelé jamais lu) :

1. **Textes libres v4** : Qwen2.5-7B rédige les 40 documents (~40 min) ;
2. **Robustesse** : le LLM Gold sur les 3 jeux perturbés (~45 min) ;
3. **RQ2** : génération directe par le LLM, 3 graines (~45 min).

**Réglages (panneau de droite)** : Accelerator **GPU T4 x2** ; Internet **On** ; Persistence : aucune.
**Lancement** : **Run All**. Un zip est écrit après chaque étape dans `/kaggle/working/resultats/` (onglet Output) : télécharger `sorties_lot_d_gold.zip` et `libres_v4.zip`.

## 1. Un seul GPU, récupérer la branche `lot-d-gold`

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
%cd /kaggle/working
if os.path.isdir('fardi-caseforge/.git'):
    !git -C fardi-caseforge pull -q origin lot-d-gold
else:
    !git clone -q -b lot-d-gold https://github.com/onsalouini/fardi-caseforge.git
%cd /kaggle/working/fardi-caseforge
!git log --oneline -1
!nvidia-smi -L

## 2. Dépendances

In [ ]:
!pip install -q -U "transformers>=4.44" accelerate bitsandbytes

## 3. Textes libres v4 : 40 documents (~40 min)
Les 2 premiers d'abord (essai) : chaque ligne doit finir par `valeurs absentes []`.

In [ ]:
!python generation/textes_libres.py rediger --limite 2
!python generation/textes_libres.py rediger

In [ ]:
import os, shutil
os.makedirs('/kaggle/working/resultats', exist_ok=True)
os.makedirs('generation/sorties', exist_ok=True)
shutil.make_archive('/kaggle/working/resultats/sorties_lot_d_gold', 'zip', 'generation', 'sorties')
if os.path.isdir('data/gold/libres_v4/brut'):
    shutil.make_archive('/kaggle/working/resultats/libres_v4', 'zip', 'data/gold', 'libres_v4')
!ls -la /kaggle/working/resultats

## 4. Robustesse du LLM : abréviations, OCR, mise en page (~45 min)

In [ ]:
for v in ('abreviations', 'ocr', 'mise_en_page'):
    !python generation/llm.py --docs generation/jeux/robustesse_{v}_dev.jsonl --graine 42 --reprendre --out generation/sorties/robustesse_llm_{v}_s42.jsonl

In [ ]:
import os, shutil
os.makedirs('/kaggle/working/resultats', exist_ok=True)
os.makedirs('generation/sorties', exist_ok=True)
shutil.make_archive('/kaggle/working/resultats/sorties_lot_d_gold', 'zip', 'generation', 'sorties')
if os.path.isdir('data/gold/libres_v4/brut'):
    shutil.make_archive('/kaggle/working/resultats/libres_v4', 'zip', 'data/gold', 'libres_v4')
!ls -la /kaggle/working/resultats

## 5. RQ2 : génération directe par le LLM, 3 graines (~45 min)

In [ ]:
for g in (13, 42, 2026):
    !python generation/llm.py --split dev --schema direct --sans-controle --graine {g} --reprendre --out generation/sorties/rq2_direct_dev_s{g}.jsonl

In [ ]:
import os, shutil
os.makedirs('/kaggle/working/resultats', exist_ok=True)
os.makedirs('generation/sorties', exist_ok=True)
shutil.make_archive('/kaggle/working/resultats/sorties_lot_d_gold', 'zip', 'generation', 'sorties')
if os.path.isdir('data/gold/libres_v4/brut'):
    shutil.make_archive('/kaggle/working/resultats/libres_v4', 'zip', 'data/gold', 'libres_v4')
!ls -la /kaggle/working/resultats